In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import plate_appearances

df = load_all_snapshots(seasons=[2024])
pa = plate_appearances(df)

# wOBA allowed per pitcher — the symmetric target to the batter score.
woba_allowed = pa.groupby("pitcher").agg(
    woba_num=("woba_value", "sum"),
    woba_den=("woba_denom", "sum"),
    pa=("woba_value", "size"),
)
woba_allowed["woba_against"] = (
    pd.to_numeric(woba_allowed["woba_num"], errors="coerce")
    / pd.to_numeric(woba_allowed["woba_den"], errors="coerce")
)

print(f"{len(woba_allowed)} pitchers")
print(woba_allowed[woba_allowed["pa"] >= 200]["woba_against"].describe().round(4).to_string())

855 pitchers
count     368.0
mean     0.3162
std      0.0372
min      0.1921
25%      0.2928
50%      0.3169
75%      0.3404
max      0.4209


In [2]:
from src.features.plate_discipline import add_discipline_flags
from src.features.batted_ball import batted_ball_events, add_quality_flags
from src.features.pitch_outcomes import add_batted_ball_types

f = add_discipline_flags(df)
bbe = add_quality_flags(batted_ball_events(df))
bbe = add_batted_ball_types(bbe)

sw = f[f["is_swing"]]
oz = f[~f["in_zone"]]

skills = pd.DataFrame({
    "pitches": f.groupby("pitcher").size(),
    "zone_pct": f.groupby("pitcher")["in_zone"].mean(),
    "whiff_pct": sw.groupby("pitcher")["is_whiff"].mean(),
    "chase_pct": oz.groupby("pitcher")["is_swing"].mean(),
    "n_swings": sw.groupby("pitcher").size(),
    "n_oz": oz.groupby("pitcher").size(),
    "gb_pct": bbe.groupby("pitcher")["is_ground_ball"].mean(),
    "barrel_pct": bbe.groupby("pitcher")["is_barrel"].mean(),
    "hard_hit_pct": bbe.groupby("pitcher")["is_hard_hit"].mean(),
    "bbe": bbe.groupby("pitcher").size(),
})

# Strikeouts and walks — xwOBA cannot see these, so they must be explicit.
ends = pa[pa["events"].notna()]
skills["k_pct"] = ends.groupby("pitcher")["events"].apply(lambda s: (s == "strikeout").mean())
skills["bb_pct"] = ends.groupby("pitcher")["events"].apply(
    lambda s: s.isin(["walk", "intent_walk"]).mean())

skills = skills.join(woba_allowed[["woba_against", "pa"]])
print(skills.describe().round(4).to_string())

         pitches  zone_pct  whiff_pct  chase_pct   n_swings       n_oz    gb_pct  barrel_pct  hard_hit_pct       bbe     k_pct    bb_pct  woba_against        pa
count   855.0000  855.0000   855.0000   853.0000   855.0000   853.0000  854.0000       854.0         854.0  854.0000  855.0000  855.0000         855.0     855.0
mean    831.1485    0.4887     0.2175     0.2724   395.7474   420.9496    0.4355      0.0811        0.3983  144.7424    0.2019    0.0928        0.3427  213.3754
std     810.3174    0.0599     0.0759     0.0615   391.6752   409.9404    0.1289      0.0604        0.1013  144.2637    0.0857    0.0594        0.0993  208.6074
min       2.0000    0.0952     0.0000     0.0000     1.0000     1.0000    0.0000         0.0           0.0    1.0000    0.0000    0.0000           0.0       1.0
25%     179.0000    0.4661     0.1859     0.2439    85.0000    91.0000    0.3711       0.054        0.3523   34.0000    0.1658    0.0625        0.2943      47.0
50%     609.0000    0.4935     0.2

In [3]:
SKILLS = ["whiff_pct", "chase_pct", "zone_pct", "gb_pct",
          "barrel_pct", "hard_hit_pct", "k_pct", "bb_pct"]

q = skills[(skills["n_swings"] >= 200) & (skills["bbe"] >= 100)
           & (skills["pa"] >= 200) & skills["woba_against"].notna()].copy()

print(f"{len(q)} qualified pitchers")
print()
print(q[SKILLS].corr().round(2).to_string())

368 qualified pitchers

              whiff_pct  chase_pct  zone_pct  gb_pct  barrel_pct  hard_hit_pct  k_pct  bb_pct
whiff_pct          1.00       0.46     -0.36   -0.09       -0.05         -0.23   0.82    0.30
chase_pct          0.46       1.00     -0.15    0.04       -0.17         -0.22   0.36   -0.31
zone_pct          -0.36      -0.15      1.00   -0.06        0.02          0.04  -0.10   -0.54
gb_pct            -0.09       0.04     -0.06    1.00       -0.48          0.14  -0.12   -0.07
barrel_pct        -0.05      -0.17      0.02   -0.48        1.00          0.42  -0.08    0.10
hard_hit_pct      -0.23      -0.22      0.04    0.14        0.42          1.00  -0.26    0.03
k_pct              0.82       0.36     -0.10   -0.12       -0.08         -0.26   1.00    0.16
bb_pct             0.30      -0.31     -0.54   -0.07        0.10          0.03   0.16    1.00


In [4]:
# whiff_pct dropped: r = 0.82 with k_pct, and Day 22 showed it is biased
# against sinker-heavy pitchers.
# zone_pct dropped: r = -0.54 with bb_pct, largely the same information.
PITCHER_SKILLS = ["k_pct", "bb_pct", "chase_pct",
                  "gb_pct", "barrel_pct", "hard_hit_pct"]

print(q[PITCHER_SKILLS].corr().round(2).to_string())
print()

from sklearn.linear_model import LinearRegression

X = q[PITCHER_SKILLS].apply(lambda s: (s - s.mean()) / s.std())
y = q["woba_against"].astype(float)

lm = LinearRegression().fit(X, y)
coefs = pd.Series(lm.coef_, index=PITCHER_SKILLS).sort_values(key=abs, ascending=False)

print("standardised coefficients (wOBA against per 1 sd):")
print(coefs.round(4).to_string())
print(f"R-squared: {lm.score(X, y):.4f}")

              k_pct  bb_pct  chase_pct  gb_pct  barrel_pct  hard_hit_pct
k_pct          1.00    0.16       0.36   -0.12       -0.08         -0.26
bb_pct         0.16    1.00      -0.31   -0.07        0.10          0.03
chase_pct      0.36   -0.31       1.00    0.04       -0.17         -0.22
gb_pct        -0.12   -0.07       0.04    1.00       -0.48          0.14
barrel_pct    -0.08    0.10      -0.17   -0.48        1.00          0.42
hard_hit_pct  -0.26    0.03      -0.22    0.14        0.42          1.00

standardised coefficients (wOBA against per 1 sd):
k_pct          -0.0196
barrel_pct      0.0106
bb_pct          0.0098
hard_hit_pct    0.0052
gb_pct          0.0018
chase_pct      -0.0007
R-squared: 0.5240


In [5]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
coef_runs, rank_runs = [], []
for tr, _ in kf.split(X):
    m = LinearRegression().fit(X.iloc[tr], y.iloc[tr])
    coef_runs.append(m.coef_)
    rank_runs.append(pd.Series(m.predict(X), index=q.index).rank())

print(pd.DataFrame(coef_runs, columns=PITCHER_SKILLS).agg(["mean", "std"]).round(4).to_string())
print()
print("min rank correlation:",
      round(pd.DataFrame(rank_runs).T.corr(method="spearman").to_numpy()[
          ~np.eye(5, dtype=bool)].min(), 3))

       k_pct  bb_pct  chase_pct  gb_pct  barrel_pct  hard_hit_pct
mean -0.0196  0.0097    -0.0007  0.0019      0.0106        0.0051
std   0.0006  0.0006     0.0012  0.0013      0.0012        0.0008

min rank correlation: 0.985


In [6]:
from src.models import whiff
wm = whiff.build()
print(wm.X_train.shape)

(200735, 31)
